# Virginia DEQ Permit Site Exploration

This notebook inspects the Virginia DEQ Title V permit listing page to understand the structure of its tables and document links. We'll capture rendered HTML via Selenium, parse the permit table, and inspect representative rows.


In [3]:
import os
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

BASE_URL = "https://www.deq.virginia.gov"
PERMIT_LISTING_URL = "https://www.deq.virginia.gov/news-info/shortcuts/permits/air/issued-title-v-permits"

print(PERMIT_LISTING_URL)


https://www.deq.virginia.gov/news-info/shortcuts/permits/air/issued-title-v-permits


In [4]:
def build_driver(headless: bool = True) -> webdriver.Chrome:
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")

    chrome_binary = os.getenv("CHROME_BINARY")
    if chrome_binary:
        options.binary_location = chrome_binary

    driver = webdriver.Chrome(options=options)
    return driver


In [5]:
driver = build_driver(headless=False)
driver.get(PERMIT_LISTING_URL)

# Wait for the permit table to render
wait = WebDriverWait(driver, 20)
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "table")))

page_source = driver.page_source
print(f"Fetched {len(page_source)} characters of HTML")


Fetched 317982 characters of HTML


In [6]:
soup = BeautifulSoup(page_source, "html.parser")
permit_tables = soup.find_all("table")

print(f"Found {len(permit_tables)} tables")
for idx, table in enumerate(permit_tables[:3]):
    headers = [th.get_text(strip=True) for th in table.find_all("th")]
    print(f"Table {idx} headers: {headers[:6]}")


Found 1 tables
Table 0 headers: ['Air Site Name', 'Registration No', 'Permit Issuance Date', 'City/County', 'Regional Office']


In [7]:
target_table = permit_tables[0]
rows = []
headers = [th.get_text(strip=True) for th in target_table.find_all("th")]

for tr in target_table.find("tbody").find_all("tr"):
    cells = tr.find_all(["td", "th"])
    values = [cell.get_text(strip=True) for cell in cells]
    link = tr.find("a")
    href = link.get("href") if link else None
    rows.append(values + [href])

columns = headers + ["Link"]
permit_df = pd.DataFrame(rows, columns=columns)
permit_df.head()


,Air Site Name,Registration No,Permit Issuance Date,City/County,Regional Office,Link
0,Yokohama Tire Manufacturing Virginia LLC,20123,04/12/2022,Salem City,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
1,Virginia Tech,20124,09/13/2021,Montgomery County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
2,RES dba Steel Dynamics Roanoke Bar Division,20131,04/24/2023,Roanoke City,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
3,Columbia Gas Transmission Corporation Gala Com...,20157,10/02/2025,Botetourt County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
4,Roanoke Cement Company,20232,12/01/2003,Botetourt County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...


In [8]:
permit_df["Permit URL"] = permit_df["Link"].apply(lambda link: urljoin(BASE_URL, link) if pd.notna(link) else None)
permit_df.drop(columns=["Link"], inplace=True)
permit_df.head(10)


,Air Site Name,Registration No,Permit Issuance Date,City/County,Regional Office,Permit URL
0,Yokohama Tire Manufacturing Virginia LLC,20123,04/12/2022,Salem City,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
1,Virginia Tech,20124,09/13/2021,Montgomery County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
2,RES dba Steel Dynamics Roanoke Bar Division,20131,04/24/2023,Roanoke City,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
3,Columbia Gas Transmission Corporation Gala Com...,20157,10/02/2025,Botetourt County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
4,Roanoke Cement Company,20232,12/01/2003,Botetourt County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
5,Celanese Acetate LLC,20304,07/18/2022,Giles County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
6,WestRock Virginia LLC - Covington,20328,07/09/2024,Covington City,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
7,Ingevity Virginia Corporation,20329,04/30/2024,Covington City,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
8,New Millennium Building Systems,20338,11/17/2020,Roanoke County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
9,Frank Chervan Incorporated - 20523,20523,03/13/2020,Roanoke City,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...


In [9]:
permit_df

,Air Site Name,Registration No,Permit Issuance Date,City/County,Regional Office,Permit URL
0,Yokohama Tire Manufacturing Virginia LLC,20123,04/12/2022,Salem City,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
1,Virginia Tech,20124,09/13/2021,Montgomery County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
2,RES dba Steel Dynamics Roanoke Bar Division,20131,04/24/2023,Roanoke City,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
3,Columbia Gas Transmission Corporation Gala Com...,20157,10/02/2025,Botetourt County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
4,Roanoke Cement Company,20232,12/01/2003,Botetourt County,Blue Ridge,https://www.deq.virginia.gov/home/showpublishe...
...,...,...,...,...,...,...
223,Rockingham County Landfill,81569,02/29/2024,Harrisonburg City,Valley,https://www.deq.virginia.gov/home/showpublishe...
224,Augusta Regional Landfill,81573,08/19/2024,Augusta County,Valley,https://www.deq.virginia.gov/home/showpublishe...
225,Evolve Manufacturing LLC,81686,07/24/2024,Frederick County,Valley,https://www.deq.virginia.gov/home/showpublishe...
226,Blue Ridge Resource Authority Landfill,81719,09/01/2023,Rockbridge County,Valley,https://www.deq.virginia.gov/home/showpublishe...
